# MCA - Multiple Correspondence Analysis : detect profiles in the population



ACM : Analyse factorielle des correspondances multiples



!!! Compléter les références !!!


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import scipy.stats as stats
import sqlite3 as sql

In [ ]:
from fanalysis.ca import CA 
from fanalysis.mca import MCA

In [ ]:
### Importer un module de fonctions crées ad hoc
##  ATTENTION : le fichier 'sparql_functions.py' doit se trouver 
#   dans un dossier qui se situe dans le chemin ('path') de recherche
#   vu par le présent carnet Jupyter afin que
#   l'importation fonctionne correctement

import sys
from importlib import reload


# Add parent directory to the path
sys.path.insert(0, '..')

### If you want to add the parent-parent directory,
sys.path.insert(0, '../..')



In [ ]:

import bivariate_library as bl
import correspondence_analysis_library as cal
import cluster_functions as cf


In [ ]:
### Use this pnéy to reload the functions if modified
print(reload(bl))
print(reload(cal))  

## Import the data and prepare the analysis

In this notebook, we use the data produced with the [bivariate analysis notebook using countries](da3-1_countries_bivariate_analysis.ipynb) and the data collected in the [da5-employer.md](../../documentation/wikidata/data-analysis/da5-employer.md) and [da5-employer.sql](../../documentation/wikidata/data-analysis/da5-employer.sql) files.



In [ ]:
csv_address='da_data/da4-AFC.csv'
df_p = pd.read_csv(csv_address)
df_p.head(3)

In [ ]:
df_p = df_p.drop(['CNTR_ID', 'CNTR_NAME', 'ISO3_CODE', 'activityYear', 'uriPlace', 'periodsActivity', 'FID', 'CNTR_ID'], axis=1)

In [ ]:
### Inspect the dataframe and 
# notably if there are missing values
df_p.info()

In [ ]:
df_p=df_p.rename(columns={'uriPer': 'person_uri'})

In [ ]:
df_p.head(2)

In [ ]:
pd.set_option('display.max_columns', None)
# Reset to default settings if needed later
# pd.reset_option('display.max_columns')

In [ ]:
### Import the new qualitative values to be added: occupation and employer class
csv_address='da_data/da5-persons-features.csv'
# Explicitly tell pandas which strings to treat as NA (exclude 'NA' from the list)
# By default, pandas treats 'NA', 'N/A', 'NaN', etc. as missing values.
# If you want to override this by specifying a custom list that does NOT include 'NA'.
## df_pfeat = pd.read_csv(csv_address, na_values=['', 'N/A', 'NULL', 'None'])
df_pfeat = pd.read_csv(csv_address)
df_pfeat.head()

In [ ]:
### Observe the 
df_pfeat.info()

In [ ]:
## Merge the two dataframes
df_pm = pd.merge(df_p,df_pfeat, on='person_uri')  
# following part was used for inspection
# , how='left')

In [ ]:
## The small number depends on the considered countries
df_pm.info()

### Manage secondary occupation

We will have to code a part of the categories because their number is too small, which makes the variable too sparse.

In [ ]:
### Group and count: secondary occupation
# We observe some dispersion that requires grouping the categories of the variable
df_count = df_pm.groupby('occupation_sec1').size()
df_count = pd.DataFrame(df_count.sort_values(ascending = False))
df_count.columns=['number']
print(len(df_count))
print(df_count.iloc[:50])



In [ ]:
## We only take the first 19 elements in the list, the more interesting ones
ldf = df_count[df_count.number > 19].index.to_list()
print(ldf)

In [ ]:
## prepare the function to code the data
def code_occupation_sec1(occup, ldf):

    """
    codes values according to list
    """

    if occup in ldf:
        output = occup
    else:
        output= 'other'
    return output        

In [ ]:
### Code the data
df_pm.occupation_sec1 = df_pm.occupation_sec1.apply(lambda x: code_occupation_sec1(x, ldf))

In [ ]:
### Group and count: secondary occupation
# We observe the results of the codind
df_count = df_pm.groupby('occupation_sec1').size()
df_count = pd.DataFrame(df_count.sort_values(ascending = False))
df_count.columns=['number']
print(df_count)


In [ ]:
df_pm.head(3)

## MCA

In [ ]:
data_cat = df_pm[['gender', 'coded_country', 'occupation_sec1', 'coded_employer']]
data_cat.head(3)


In [ ]:
DActives=data_cat

In [ ]:
# We inspect the table that will be analysed
p = DActives.shape[1]
#nombre d'observations
n = DActives.shape[0]
print('Number of variables:', p, ' -- Nomber of individuals (rows):', n)
#codage en 0/1


### Complete disjunctive table 

Sometimes also referred to as a dummy variable matrix or indicator matrix.

In the context of Multiple Correspondence Analysis (MCA), this table is used to transform qualitative (categorical) variables into a quantitative format suitable for analysis. Each category of a qualitative variable becomes a separate binary column (0 or 1), indicating the absence or presence of that category for each individual.

In [ ]:
## Complete disjunctive table
X = pd.get_dummies(DActives,prefix='',prefix_sep='')*1
X.head(3)

In [ ]:
### Nombre total de modalités, toute variable confondue
M = X.shape[1]
print('Number of categories:', M)
#nombre max de facteurs
Hmax = M-p
print('Maximum number of factors:', Hmax)

In [ ]:
Xm = X.copy(deep=True)
#Total sum per row: 
Xm.loc[:,'Total'] = Xm.sum(axis=1)
#Total sum per column: 
Xm.loc['Total',:] = Xm.sum(axis=0)
Xm.tail()

## MCA

The Complete disjoint table is created interanlly by the software

In [ ]:

acm = MCA(row_labels=DActives.index,var_labels=DActives.columns)
acm.fit(DActives.values)




In [ ]:
eig = pd.DataFrame(acm.eig_).transpose()
eig.columns=['contribution','freq','freq_cumulee']

print('Number of factors (approximation tables):', len(eig), '\n')
print(eig.head(), eig.tail())


In [ ]:
#éboulis des v.p.
fix,ax = plt.subplots(figsize=(5,5))
ax.plot(range(1,Hmax+1),acm.eig_[0],".-")
ax.set_xlabel("Nb. facteurs")
ax.set_ylabel("Val. propres")
plt.title("Eboulis des valeurs propres")
#seuil - Règle de Kaiser
ax.plot([1,Hmax],[1/p,1/p],"r--",linewidth=1)
plt.show()

*Diagramme d'éboulis*. Représentation graphique ayant pour but d'identifier un point d'inflexion dans une courbe de la variance. Le nom donné à ce type de graphique vient de la ressemblance de la courbe avec le profil des éboulis (scree) au bas d'une falaise. [DataFranca, Diagramme d'éboulis, 2024](https://datafranca.org/wiki/index.php?title=Diagramme_d%27%C3%A9boulis&oldid=93502)

In [ ]:

fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(12,3))

eig.iloc[:,0].plot(kind='bar', ax=axes[0], title='Eigenvalue des axes')
eig.iloc[:,1].plot(kind='bar', ax=axes[1], title="Proportion de l'eigenvalue ")
eig.iloc[:,2].plot(kind='bar', ax=axes[2], title="Frequence cumulative de l'eigenvalue ")
# Met les valeurs xticks en vertical
fig.autofmt_xdate(rotation=0)
plt.show()

In [ ]:
# Mapping des points colonnes
acm.mapping_col(num_x_axis=1, num_y_axis=2, figsize=(20,20))

In [ ]:
# Mapping des points lignes: individus
acm.mapping_row(num_x_axis=1, num_y_axis=2, figsize=(20,20))

## Représenter les individus

In [ ]:
# Mapping simultané des points lignes et colonnes
# Les paramètres de la méthode mapping indiquent que ce sont les axes 1 et 2 qui sont ici représentés
acm.mapping(num_x_axis=1, num_y_axis=2, figsize=(20,20), )

In [ ]:
### Inspect individuals
df_pm.loc[[77, 42, 2, 50, 111]]

## Une possibilité d'interprétation: distance des individus par rapport au profil moyen

In [ ]:
#Profil individu moyen
ind_moy = np.sum(X.values,axis=0)/(n*p)
print(ind_moy)

In [ ]:
### Ajouter une colonne avec la distance chi-2 de chaque individu par rapport à l'individu moyen
# pour chaque individu: les individus plus éloignés sont plus rares
X['dist_org'] = X.apply(lambda x: round(np.sum(1/ind_moy*(x/p-ind_moy)**2),4), raw=True, axis=1)
X['dist_org']

In [ ]:

### Distribution des distances à l'individu moyen

sns.set_theme(style="whitegrid",rc={"figure.figsize":(12,2)} )


a = X['dist_org']

print(a.describe())

# ax = sns.boxplot(x=a)
ax = sns.violinplot(x=a)

### Noter que au delà des limites les valeurs sont coupées car postulées
ax.set_xlim(left=min(a), right=max(a))

plt.show()

In [ ]:
### Individus proches du profil moyen
#  donc fréquents
i = X[(X.dist_org<8.5) & (X.dist_org>8.3)]
print(len(i))
df_pm.loc[i.index][30:35]

In [ ]:
### Individus moyennements distants du profil moyen
sel = X[(X.dist_org>8.3) & (X.dist_org <8.4)]   ### [X.dist_org<2.5] proches du prof. m.
print(len(sel))
df_pm.loc[sel.index][30:35]

In [ ]:
### Individus très distants du profil moyen
#  donc rares
seld = X[X.dist_org>200]
print(len(seld))
df_pm.loc[seld.index][-5:]

## Add the activity periods as supplementary illustrative values

In [ ]:
#isoler les variables supplémentaires (ou illustratives)
df_suppl = df_pm[['periodsActivity']]
print(df_suppl.columns)
df_suppl.head(2)

In [ ]:
info_lig = acm.row_topandas()
info_lig[:2]

In [ ]:
## We merge the two tables.
# Because they have the same order, the coordinates are added to the right individuals
df_supp_lignes = df_suppl.merge(info_lig, left_index=True, right_index=True)
df_supp_lignes.head(2)

### Plot dim1-dim2

In [ ]:
#positionnement d'individus
coord = df_supp_lignes[['periodsActivity','row_coord_dim1', 'row_coord_dim2']].copy()
print(len(coord))
coord.head(2)


In [ ]:

#moyennes conditionnelles - Livre, page 341
coord_fact = pd.pivot_table(data=coord,values=['row_coord_dim1', 'row_coord_dim2'],index='periodsActivity',aggfunc='mean')
coord_fact


In [ ]:

#corrigés par la racine carrée des valeurs propres
# computed barycenters on 2 dimensions (row_coord_dim1 and row_coord_dim2): [:2]
coord_fact = coord_fact/np.sqrt(acm.eig_[0][:2])
print(coord_fact)

In [ ]:
MCA.mapping_col = cal.custom_mapping_col

In [ ]:


# Create your own figure and axes
fig, ax = plt.subplots(figsize=(40, 40))

# Plot ACM on ax
acm.mapping_col(num_x_axis=1, num_y_axis=2, ax=ax, short_labels=False)

# Add your additional layer

# Add illustrative variable modalities (e.g., Profession)
ax = plt.gca()  # Get current axis
for i in range(coord_fact.shape[0]):
    ax.text(coord_fact.iloc[i, 0], coord_fact.iloc[i, 1], coord_fact.index[i],
            color="darkgreen", fontsize=20, ha='center', va='center')

plt.title("ACM with Illustrative Variables (Profession)")
plt.show()


## Prepare file for cluster analysis

Given that the interpretation of the MCA is quite difficult, given the sparse nature of the variables, we will test the [K-means clustering method](da5-MCA-clusters.ipynb) on the results of the MCA in order to detect significant profiles.

We therefore export here the coded and prepared data to a [dedicated CSV file](da_data/da5-MCA-clusters.csv).

In [ ]:
df_pm.head(3)

In [ ]:
file_address='da_data/da5-MCA-clusters.csv'
df_pm.to_csv(file_address, index=False)

## Split the generations and analyse significant ones

In [ ]:
activity_period='1990-2010'
df_pm[df_pm.periodsActivity==activity_period].head(3)

## MCA

In [ ]:
data_cat = df_pm[df_pm.periodsActivity==activity_period][['gender', 'coded_country', 'occupation_sec1', 'coded_employer']]
data_cat.head(3)


In [ ]:
DActives=data_cat

In [ ]:
acm = MCA(row_labels=DActives.index,var_labels=DActives.columns)
acm.fit(DActives.values)

In [ ]:
eig = pd.DataFrame(acm.eig_).transpose()
eig.columns=['contribution','freq','freq_cumulee']

print('Number of factors (approximation tables):', len(eig), '\n')
print(eig.head(), eig.tail())


In [ ]:

fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(12,3))

eig.iloc[:,0].plot(kind='bar', ax=axes[0], title='Eigenvalue des axes')
eig.iloc[:,1].plot(kind='bar', ax=axes[1], title="Proportion de l'eigenvalue ")
eig.iloc[:,2].plot(kind='bar', ax=axes[2], title="Frequence cumulative de l'eigenvalue ")
# Met les valeurs xticks en vertical
fig.autofmt_xdate(rotation=0)
plt.show()

In [ ]:
# Mapping des points colonnes
acm.mapping_col(num_x_axis=1, num_y_axis=1, figsize=(20,20))

## Représenter les individus

In [ ]:
# Mapping simultané des points lignes et colonnes
# Les paramètres de la méthode mapping indiquent que ce sont les axes 1 et 2 qui sont ici représentés
acm.mapping(num_x_axis=1, num_y_axis=2, figsize=(20,20), )

In [ ]:
### Inspect individuals
df_pm.loc[[20, 45, 80, 120]]